# Yolo

In [ ]:
!pip install -q ultralytics gdown

import os
import shutil
import gdown
import glob
import numpy as np
from ultralytics import YOLO

WORKING_DIR = '/kaggle/working'
EXTRACTED_DIR = os.path.join(WORKING_DIR, 'AI_Tourist_Dataset')
ZIP_NAME = 'AI_Tourist_Dataset_10Classes.zip'
ZIP_PATH = os.path.join(WORKING_DIR, ZIP_NAME)

file_id = '1tp-EaiiatkWasCjAlMYCMdHx29eyNBEr'
url = f'https://drive.google.com/uc?id={file_id}'

if not os.path.exists(ZIP_PATH):
    print(f"Downloading {ZIP_NAME} from Google Drive...")
    gdown.download(url, ZIP_PATH, quiet=False, fuzzy=True)

if os.path.exists(EXTRACTED_DIR):
    shutil.rmtree(EXTRACTED_DIR)

!unzip -q {ZIP_PATH} -d {EXTRACTED_DIR}

nested_folder = os.path.join(EXTRACTED_DIR, 'AI_Tourist_Dataset_10Classes')
macosx_folder = os.path.join(EXTRACTED_DIR, '__MACOSX')

if os.path.exists(macosx_folder):
    shutil.rmtree(macosx_folder)

if os.path.exists(nested_folder):
    for item in os.listdir(nested_folder):
        shutil.move(os.path.join(nested_folder, item), os.path.join(EXTRACTED_DIR, item))
    os.rmdir(nested_folder)

print(f"Data ready in: {EXTRACTED_DIR}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.5 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Downloading...
From (original): https://drive.google.com/uc?id=1tp-EaiiatkWasCjAlMYCMdHx29eyNBEr
From (redirected): https://drive.google.com/uc?id=1tp-EaiiatkWasCjAlMYCMdHx29eyNBEr&confirm=t&uuid=6e6c7ff1-9158-4d40-91df-c73e263e4b7a
To: /kaggle/working/AI_Tourist_Dataset_10Classes.zip
100%|██████████| 3.12G/3.12G [00:27<00:00, 113MB/s] 


Data ready in: /kaggle/working/AI_Tourist_Dataset


In [ ]:
model = YOLO('yolov8n-cls.pt')

results = model.train(
    data=EXTRACTED_DIR,
    epochs=20,
    imgsz=224,
    batch=32,
    project=WORKING_DIR,
    name='yolo_monument_cls'
)

Ultralytics 8.4.45 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/AI_Tourist_Dataset, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_monument_cls, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mas

In [ ]:
best_model_path = os.path.join(WORKING_DIR, 'yolo_monument_cls', 'weights', 'best.pt')
best_model = YOLO(best_model_path)

test_dir = os.path.join(EXTRACTED_DIR, 'test')
test_images = glob.glob(os.path.join(test_dir, '*/*.jpg')) + glob.glob(os.path.join(test_dir, '*/*.png'))

confidences = []

print(f"Running inference on {len(test_images)} test images to calculate average confidence...")

for img_path in test_images:
    result = best_model(img_path, verbose=False)[0]

    top1_conf = result.probs.top1conf.item()
    confidences.append(top1_conf)

average_confidence = np.mean(confidences)
print("-" * 30)
print(f"Average Confidence Score: {average_confidence:.4f}")
print(f"Suggested Backend Threshold: {average_confidence:.4f}")
print("-" * 30)

Running inference on 28 test images to calculate average confidence...
------------------------------
Average Confidence Score: 0.9993
Suggested Backend Threshold: 0.9993
------------------------------
